# Skin Mamba Lab · Kaggle T4

选择 GPU T4。设置路径和实验参数；测试只输出三个官方完整测试集（full），不生成clean子集成绩。PH²全部200张仅测试。每epoch保存last.pt。

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
CODE_INPUT = Path('/kaggle/input/your-code/skin_mamba_lab')
DATA_ROOT = Path('/kaggle/input/datasets/nero20260505/data-20260925')
WORK = Path('/kaggle/working/skin_mamba_lab')
assert (CODE_INPUT / 'skinmamba').is_dir(), '请设置真实CODE_INPUT'
assert (DATA_ROOT / 'isic2018/train/images').is_dir(), '请设置真实DATA_ROOT'
if not WORK.exists():
    shutil.copytree(CODE_INPUT, WORK)
os.chdir(WORK)
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)
print('code:',WORK,'data:',DATA_ROOT)

## 安装官方依赖并验证GPU
安装脚本保留Kaggle自带torch，可能需要编译CUDA扩展。失败时查看reports/kaggle_install.json，不能在缺少kernel时开始正式实验。默认FP32；若使用AMP，先另外执行verify_cuda.py --precision all。

In [ ]:
run('scripts/install_kaggle.py')
run('scripts/verify_cuda.py','--precision','fp32')

In [ ]:
run('-m','skinmamba','audit','--data-root',DATA_ROOT)

## 选择实验、epoch和seed
修改下一格的EXPERIMENT（baseline/main/ablation）、ABLATION、EPOCHS、SEED和SOURCE。main对应当前几何间距校准原型。train仅训练及验证；改为run会在完成后自动测试。1 epoch用于流程检查。续训时保持原模型配置、计划总轮数与seed。

In [ ]:
EXPERIMENT = 'baseline'  # 'baseline' / 'main' / 'ablation'
ABLATION = 'sampling_only'  # 仅EXPERIMENT='ablation'时使用；也可选constant_scale等
EPOCHS = 1
SEED = 2026
SOURCE = 'isic2018'  # 'isic2017' / 'isic2018'
CONFIG = f'configs/baseline_{SOURCE}.yaml'
assert EXPERIMENT in ('baseline', 'main', 'ablation')
assert SOURCE in ('isic2017', 'isic2018')
label = ABLATION if EXPERIMENT == 'ablation' else EXPERIMENT
RUN_DIR = Path(f'/kaggle/working/runs/{SOURCE}_{label}_s{SEED}')
RESUME = None  # 新会话可填 /kaggle/input/previous-output/.../last.pt
selection = ['--ablation', ABLATION] if EXPERIMENT == 'ablation' else [f'--{EXPERIMENT}']
cmd = ['-m','skinmamba','train', *selection, '--epoch', EPOCHS, '--seed', SEED,
       '--config',CONFIG,'--data-root',DATA_ROOT,'--run-dir',RUN_DIR]
if RESUME:
    cmd += ['--resume',RESUME]
run(*cmd)


## 冻结方案后做最终测试
对best.pt评估ISIC2017、ISIC2018、PH²三个完整测试集，输出full结果、CSV/JSON和可选掩码，不生成clean成绩。1 epoch仅验证代码流程；正式结果需完成计划训练。

In [ ]:
run('-m','skinmamba','evaluate','--checkpoint',RUN_DIR/'best.pt','--data-root',DATA_ROOT,'--save-predictions')
import pandas as pd
pd.read_csv(RUN_DIR/'evaluation/summary.csv')

In [ ]:
run('-m','skinmamba','aggregate','--root','/kaggle/working/runs','--output','/kaggle/working/runs/aggregate.csv')

结束前 Save Version 保存输出，保留last.pt、best.pt、配置、数据审计和逐图结果。当前代码的CPU测试不能代表T4测试通过；实际cuda_verification.json及profile.json是GPU核验依据。